# Session 3: mass changes, and expenditure changes with it
**A. Real-world problem.** The 7700 kcal-per-kg rule treats energy use as fixed. But a
heavier body uses more energy, so a constant surplus raises mass more and more slowly: mass
approaches a new steady state. We simulate this, check it against a closed-form answer,
compare it with the naive rule, and measure how much uncertain inputs blur the result.
(General education only, not medical or dietary advice.)

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

RHO = 7700.0   # kcal per kg, rough short-term rule

## Step 1: expenditure that depends on mass
With Mifflin-St Jeor, E(M) = PAL x (10 M + 6.25 h - 5 a + s). That is a straight line
E = A M + B (A in kcal/day per kg, B in kcal/day). Extra activity per kg of body mass (for
example walking) simply adds to A.

In [ ]:
def exp_params(pal, height_cm, age, sex, extra_per_kg=0.0, scale=1.0):
    """Returns (A, B) so that expenditure E(M) = A*M + B (kcal/day).

    scale multiplies the BMR (used for uncertainty: 1.1 means BMR is 10 percent higher)."""
    s = 5.0 if sex == "male" else -161.0
    A = scale * pal * 10.0 + extra_per_kg
    B = scale * pal * (6.25 * height_cm - 5.0 * age + s)
    return A, B


ASHA = dict(m0=60.0, height_cm=165.0, age=20, sex="female", pal=1.55)   # made up

## Step 2: the model and its exact answer
dM/dt = (I - E(M)) / RHO = (I - A M - B) / RHO. This is a linear equation. Its solution is

    M(t) = M_ss + (M0 - M_ss) exp(-t / tau),   M_ss = (I - B) / A,   tau = RHO / A

so mass moves toward M_ss and the speed is proportional to the distance left. We solve it
numerically (Euler, step dt days) and compare with this formula.

In [ ]:
def steady_state(intake, A, B):
    return (intake - B) / A


def time_constant(A):
    """tau in days."""
    return RHO / A


def exact_mass(m0, intake, A, B, t_days):
    mss = steady_state(intake, A, B)
    return mss + (m0 - mss) * np.exp(-np.asarray(t_days, dtype=float) * A / RHO)


def simulate(m0, intake, A, B, days, dt=1.0):
    """Euler solution. Returns (t, M) with t in days from 0 to days."""
    n = int(round(days / dt))
    t = np.arange(n + 1) * dt
    m = np.empty(n + 1)
    m[0] = m0
    for i in range(n):
        m[i + 1] = m[i] + dt * (intake - (A * m[i] + B)) / RHO
    return t, m


def naive_mass(m0, intake, A, B, t_days):
    """The 7700 rule: expenditure frozen at its starting value."""
    e0 = A * m0 + B
    return m0 + (intake - e0) * np.asarray(t_days, dtype=float) / RHO

## Step 3: uncertainty by Monte Carlo
Suppose Asha logs her food and plans to eat exactly what her estimate says she uses, so the
prediction is "no change". But logging can be off by up to 10 percent, and the BMR estimate
can be off by up to 10 percent. We draw both errors at random (uniform), run the exact solution
for each draw and look at the spread of the 12-week mass change.

In [ ]:
def monte_carlo(m0, logged_intake, A_nom, B_nom, days, n=5000, intake_err=0.10, bmr_err=0.10, seed=2024):
    """Mass change (kg) after `days` for n random error draws. Returns an array of length n."""
    rng = np.random.default_rng(seed)
    ui = rng.uniform(-intake_err, intake_err, n)
    ub = rng.uniform(-bmr_err, bmr_err, n)
    A = A_nom * (1.0 + ub)
    B = B_nom * (1.0 + ub)
    true_intake = logged_intake * (1.0 + ui)
    return exact_mass(m0, true_intake, A, B, days) - m0

## Step 4: a failure case
Two common logging errors: (a) trusting a label that is rounded (here to the nearest 10 kcal,
so each label is off by at most 5), and (b) underestimating portions. Compare their size.

In [ ]:
def rounding_worst_case(labels_per_day, step=10.0):
    """Largest possible daily error (kcal) from rounding each label to the nearest `step`."""
    return labels_per_day * step / 2.0


def portion_error(logged_intake, underestimate):
    """True intake if portions were underestimated by this fraction (0.2 = logged 20 percent too low)."""
    return logged_intake / (1.0 - underestimate)

## Step 5: a mini-project helper for a made-up planner
Net energy of walking above resting: (MET - 1) x mass x hours, with MET = 3.5 for walking
(approximate textbook value). Per kg of mass it is 2.5 x hours.

In [ ]:
WALK_NET_MET = 2.5   # (3.5 - 1)


def plan_balance(intake, walk_hours, m0=60.0, pal=1.2, height_cm=165.0, age=20, sex="female"):
    """Daily balance (kcal) at the start for a meal-and-walk plan, sedentary base plus walking."""
    A, B = exp_params(pal, height_cm, age, sex, extra_per_kg=WALK_NET_MET * walk_hours)
    return intake - (A * m0 + B)


def find_plans(meals=(1400.0, 1500.0, 1600.0, 1700.0), walks=(0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5), lo=-225.0, hi=-175.0):
    """All (meal kcal, walk hours) pairs whose starting daily balance lies in [lo, hi]."""
    return [(m, w) for m in meals for w in walks if lo <= plan_balance(m, w) <= hi]


def plan_outcome(intake, walk_hours, days=84, m0=60.0):
    """Dynamic-model mass change (kg) over `days` for the plan (sedentary base plus walking)."""
    A, B = exp_params(1.2, 165.0, 20, "female", extra_per_kg=WALK_NET_MET * walk_hours)
    return float(exact_mass(m0, intake, A, B, days) - m0)


def plan_spread(intake, walk_hours, days=84, m0=60.0, n=5000, seed=2024):
    """5th and 95th percentile of the mass change when the logged intake and BMR are each off by up to 10 percent."""
    A, B = exp_params(1.2, 165.0, 20, "female", extra_per_kg=WALK_NET_MET * walk_hours)
    d = monte_carlo(m0, intake, A, B, days, n=n, seed=seed)
    return tuple(np.percentile(d, [5, 95]))

## Step 6: see it
Left: dynamic model against the naive line, with the exact solution. Right: spread of the
12-week result from the Monte Carlo draws.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    A, B = exp_params(ASHA["pal"], ASHA["height_cm"], ASHA["age"], ASHA["sex"])
    E0 = A * ASHA["m0"] + B
    I = 2400.0
    print(f"A = {A:.2f} kcal/day per kg, B = {B:.2f}, E0 = {E0:.1f}, tau = {time_constant(A):.0f} days")
    print(f"steady state at {I:.0f} kcal/day: {steady_state(I, A, B):.2f} kg")
    t, m = simulate(ASHA["m0"], I, A, B, 730)
    i84 = 84
    print(f"12 weeks: dynamic {m[i84]:.2f}, exact {float(exact_mass(60, I, A, B, 84)):.2f}, naive {float(naive_mass(60, I, A, B, 84)):.2f} kg")
    print(f"2 years:  dynamic {m[730]:.2f}, naive {float(naive_mass(60, I, A, B, 730)):.2f} kg")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.plot(t / 7, m, label="dynamic model (Euler)")
    ax1.plot(t / 7, exact_mass(60, I, A, B, t), "k:", label="closed form")
    ax1.plot(t / 7, naive_mass(60, I, A, B, t), "--", label="naive 7700 rule")
    ax1.axhline(steady_state(I, A, B), color="gray", lw=0.8)
    ax1.axvline(12, color="gray", ls=":")
    ax1.set_xlabel("time (weeks)")
    ax1.set_ylabel("body mass (kg)")
    ax1.set_title("Mass approaches a steady state (grey line)")
    ax1.grid(alpha=0.3)
    ax1.legend()

    logged = E0
    d = monte_carlo(60.0, logged, A, B, 84)
    p5, p50, p95 = np.percentile(d, [5, 50, 95])
    print(f"Monte Carlo (logged intake = {logged:.0f}): 5th {p5:.2f}, median {p50:.2f}, 95th {p95:.2f} kg, sd {d.std():.2f}")
    print(f"label rounding worst case for 8 labels: {rounding_worst_case(8):.0f} kcal/day")
    print(f"20 percent under-logging of {logged:.0f} kcal: true intake {portion_error(logged, 0.2):.0f} kcal/day")
    ax2.hist(d, bins=40, color="#457b9d")
    ax2.axvline(p5, color="k", ls="--")
    ax2.axvline(p95, color="k", ls="--", label="5th and 95th percentile")
    ax2.set_xlabel("12-week mass change (kg)")
    ax2.set_ylabel("number of draws (of 5000)")
    ax2.set_title("Logging and BMR errors of up to 10 percent")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig.tight_layout()
    fig.savefig(OUT / "dynamic_and_spread.png", dpi=120)

    # the mini-project table
    print("walk (h)  balance with 1500 kcal meals")
    for h in (0.0, 0.25, 0.5, 0.75, 1.0):
        print(f"{h:5.2f}    {plan_balance(1500.0, h):8.1f}")
    print("passing plans:", find_plans())
    print("outcome (1500, 0.25):", round(plan_outcome(1500, 0.25), 2), plan_spread(1500, 0.25))
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Run the Euler solution with dt = 0.5 and dt = 2 days and compare the errors at 12 weeks with the closed form.
2. **Stretch.** Change the BMR error in `monte_carlo` to 5 percent and describe how the spread changes.
3. **Explain.** Explain, without equations, why a 12-week weigh-in cannot tell an exact plan from a slightly different one.

## Assessment
- **A. Conceptual.** Why does the dynamic curve flatten while the naive line does not?
- **B. Numerical.** For Asha, what steady-state mass does an intake of 2200 kcal a day give, and what fraction of the way to the steady state has the mass travelled after one time constant?
- **C. Graph and simulation interpretation.** Reading the first figure: by how much do the naive and dynamic answers differ at 12 weeks and at 2 years (intake 2400)? What does this tell you about using the 7700 rule?
- **D. Design and optimisation challenge.** Design: a planner for the made-up person Asha (sedentary base, 60 kg) wants a starting daily balance between -225 and -175 kcal using meal plans of 1400, 1500, 1600 or 1700 kcal and walking in 15 minute steps up to 1.5 hours. Which plans pass, which is the least walking, and can a 12-week weigh-in confirm it? State a trade-off.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    a = L.ASHA
    A, B = L.exp_params(a["pal"], a["height_cm"], a["age"], a["sex"])
    E0 = A * a["m0"] + B
    checks.append(("Asha: A = 15.5 kcal/day per kg and B = 1193.89 kcal/day", abs(A - 15.5) < 1e-9 and round(B, 2) == 1193.89))
    checks.append(("Asha's starting expenditure is 2123.9 kcal/day", round(E0, 1) == 2123.9))
    checks.append(("time constant RHO / A is about 497 days", round(L.time_constant(A)) == 497))
    I = 2400.0
    t, m = L.simulate(60.0, I, A, B, 84)
    ex = L.exact_mass(60.0, I, A, B, t)
    checks.append(("Euler (dt = 1 day) matches the closed-form exponential within 0.01 kg over 12 weeks", float(np.max(np.abs(m - ex))) < 0.01))
    _, m_half = L.simulate(60.0, I, A, B, 84, dt=0.5)
    e1 = abs(m[-1] - ex[-1])
    e2 = abs(m_half[-1] - ex[-1])
    checks.append(("halving the time step roughly halves the Euler error (first-order method)", 1.7 < e1 / e2 < 2.3))
    checks.append(("worked example: eating 2400 kcal/day gives a steady state of 77.81 kg",
                   round(float(L.steady_state(I, A, B)), 2) == 77.81))
    checks.append(("worked example: after 12 weeks the dynamic model gives 62.77 kg and the naive rule 63.01 kg",
                   round(float(ex[-1]), 2) == 62.77 and round(float(L.naive_mass(60.0, I, A, B, 84)), 2) == 63.01))
    t2, m2 = L.simulate(60.0, I, A, B, 730)
    checks.append(("after 2 years the dynamic model gives 73.72 kg but the naive rule 86.18 kg",
                   round(float(m2[-1]), 2) == 73.72 and round(float(L.naive_mass(60.0, I, A, B, 730)), 2) == 86.18))
    checks.append(("the dynamic curve never passes the steady state (monotone approach)",
                   bool(np.all(np.diff(m2) > 0)) and float(m2[-1]) < float(L.steady_state(I, A, B))))
    checks.append(("intake equal to expenditure keeps mass at 60 kg (edge case)", abs(float(L.exact_mass(60.0, E0, A, B, 500)) - 60.0) < 1e-9))
    checks.append(("naive and dynamic agree at time zero", float(L.naive_mass(60.0, I, A, B, 0)) == 60.0 == float(L.exact_mass(60.0, I, A, B, 0))))
    d = L.monte_carlo(60.0, E0, A, B, 84)
    p5, p50, p95 = np.percentile(d, [5, 50, 95])
    checks.append(("Monte Carlo: the 5th and 95th percentiles of the 12-week change are about -2.87 and +2.88 kg",
                   abs(p5 + 2.87) < 0.1 and abs(p95 - 2.88) < 0.1))
    checks.append(("Monte Carlo: the median is near zero and the standard deviation is about 1.74 kg",
                   abs(p50) < 0.1 and abs(d.std() - 1.74) < 0.1))
    d0 = L.monte_carlo(60.0, E0, A, B, 84, intake_err=0.0, bmr_err=0.0, n=100)
    checks.append(("with no input error the spread is zero (edge case)", float(np.max(np.abs(d0))) < 1e-9))
    checks.append(("rounding 8 labels to the nearest 10 kcal is at most 40 kcal/day off", L.rounding_worst_case(8) == 40.0))
    true_i = L.portion_error(E0, 0.2)
    checks.append(("logging 20 percent too little hides about 531 kcal/day (true intake 2655)", round(true_i) == 2655 and round(true_i - E0) == 531))
    gain = float(L.exact_mass(60.0, true_i, A, B, 84)) - 60.0
    checks.append(("that hidden 531 kcal/day is a gain of about 5.33 kg in 12 weeks, far from the predicted 0", abs(gain - 5.33) < 0.02))
    checks.append(("hand calculation: 1500 kcal meals with no walking leave a balance of -144.3 kcal/day", round(L.plan_balance(1500.0, 0.0), 1) == -144.3))
    checks.append(("a quarter hour of walking adds 37.5 kcal/day, so the balance is -181.8", round(L.plan_balance(1500.0, 0.25), 1) == -181.8))
    checks.append(("plans in the window -225 to -175 kcal/day are (1500, 0.25), (1500, 0.5), (1600, 1.0)",
                   L.find_plans() == [(1500.0, 0.25), (1500.0, 0.5), (1600.0, 1.0)]))
    checks.append(("the 1500 kcal and 15 minute walk plan changes mass by -1.85 kg in 12 weeks", round(L.plan_outcome(1500.0, 0.25), 2) == -1.85))
    lo, hi = L.plan_spread(1500.0, 0.25)
    checks.append(("its 5th to 95th percentile spread is about -4.03 to +0.33 kg, so a 10 percent error can hide the whole effect",
                   abs(lo + 4.03) < 0.1 and abs(hi - 0.33) < 0.1 and hi > 0))
    checks.append(("Asha's surplus at 2400 kcal/day is 276.1 kcal/day", round(2400.0 - E0, 1) == 276.1))
    checks.append(("design: an intake of 2200 kcal/day gives a steady state of 64.91 kg", round(float(L.steady_state(2200.0, A, B)), 2) == 64.91))
    checks.append(("after one time constant the mass has covered 63.2 percent of the way", round(1.0 - float(np.exp(-1.0)), 3) == 0.632
                   and abs((float(L.exact_mass(60.0, I, A, B, L.time_constant(A))) - 60.0) / (float(L.steady_state(I, A, B)) - 60.0) - 0.632) < 1e-3))
    gap2y = float(L.naive_mass(60.0, I, A, B, 730)) - float(L.exact_mass(60.0, I, A, B, 730))
    gap12 = float(L.naive_mass(60.0, I, A, B, 84)) - float(L.exact_mass(60.0, I, A, B, 84))
    checks.append(("the naive and dynamic answers differ by 0.24 kg at 12 weeks but 12.46 kg at 2 years", round(gap12, 2) == 0.24 and round(gap2y, 2) == 12.46))
    checks.append(("snippet numbers: Mifflin for Asha is 1370.25 kcal/day and 5/60/20 g label is 440 kcal",
                   abs(10 * 60 + 6.25 * 165 - 5 * 20 - 161 - 1370.25) < 1e-9 and 4 * 5 + 4 * 60 + 9 * 20 == 440))
    A_hi, B_hi = L.exp_params(a["pal"], a["height_cm"], a["age"], a["sex"], scale=1.1)
    checks.append(("a BMR 10 percent too high raises A and B by 10 percent", abs(A_hi - 1.1 * A) < 1e-9 and abs(B_hi - 1.1 * B) < 1e-9))
    checks.append(("the span between the 5th and 95th percentile at 12 weeks is about 5.8 kg", abs((p95 - p5) - 5.75) < 0.15))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
